In [52]:
! pip install pandas numpy matplotlib requests thefuzz duckdb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
import os
import pandas as pd
from thefuzz import fuzz
import numpy as np
from scipy import stats


# 3.1 Data Ingestion & Profiling

### Repeatable ingestion pipeline that downloads and extracts city data

In [54]:
def ingest_city_data(directory_path="."):

  datasets = {}
  expected_files = {
      'listings': 'listings.csv',
      'calendar': 'calendar.csv',
      'reviews': 'reviews.csv'
  }
  for key, filename in expected_files.items():
    file_path = os.path.join(directory_path, filename)
    if os.path.exists(file_path):
      print(f"Loading '{filename}'...")
      datasets[key] = pd.read_csv(file_path, low_memory=False)
      print(f"   SUCCESS: Loaded {key} | Shape: {datasets[key].shape}")
    else:
      print(f"   WARNING: File '{filename}' not found in '{directory_path}'.")
      datasets[key] = None
  return datasets
  
raw_data = ingest_city_data()
# print(raw_data)

Loading 'listings.csv'...
   SUCCESS: Loaded listings | Shape: (16107, 90)
Loading 'calendar.csv'...
   SUCCESS: Loaded calendar | Shape: (5879060, 5)
Loading 'reviews.csv'...
   SUCCESS: Loaded reviews | Shape: (992991, 6)


### Profile each dataset & Generate a data quality report summarizing findings

In [55]:
def profile_dataset(df,name):
    
    if df is None:
        return None
    
    total_rows = len(df)
    profile_records = []
    
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_rate = (null_count / total_rows) * 100
        cardinality = df[col].nunique()
        dtype = str(df[col].dtype)
        
        profile_records.append({
            'Dataset': name,
            'Column' : col,
            'DataType' : dtype,
            'Cardinality' : cardinality,
            'NullCount' : null_count,
            'NullRate' : round(null_rate, 2)
        })    
    return pd.DataFrame(profile_records)  

# #Call the function and Print the output
# calendar_df = pd.read_csv('calendar.csv')
# calendar_profile = profile_dataset(calendar_df, 'calendar')
# display(calendar_profile)   

profiles = {}
dq_summary = []

for name, df in raw_data.items():
    if df is not None:
        profiles[name] = profile_dataset(df, name)
        
        total_cells = df.size 
        total_nulls = df.isnull().sum().sum()
        overall_null_rate = (total_nulls / total_cells) * 100
        memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
        
        dq_summary.append({
            'Dataset' : name,
            'Total Rows': df.shape[0],
            'Total Columns' : df.shape[1],
            'Global Null Rate (%)': round(overall_null_rate, 2),
            'Memory Footprint': f"{memory_mb:.2f} MB"
        })
        print(f"\n--- Profiling Snippet: {name.upper()} (Top 5 Columns) ---")
        display(profiles[name].head(5))
        
print("\n--- GLOBAL DATA QUALITY REPORT ---")
dq_report_df = pd.DataFrame(dq_summary)
display(dq_report_df)
print("\n" + "="*80 + "\n")


--- Profiling Snippet: LISTINGS (Top 5 Columns) ---


,Dataset,Column,DataType,Cardinality,NullCount,NullRate
0,listings,id,int64,16107,0,0.0
1,listings,listing_url,object,16107,0,0.0
2,listings,scrape_id,int64,1,0,0.0
3,listings,last_scraped,object,2,0,0.0
4,listings,source,object,2,0,0.0



--- Profiling Snippet: CALENDAR (Top 5 Columns) ---


,Dataset,Column,DataType,Cardinality,NullCount,NullRate
0,calendar,listing_id,int64,16107,0,0.0
1,calendar,date,object,366,0,0.0
2,calendar,available,object,2,0,0.0
3,calendar,minimum_nights,int64,98,0,0.0
4,calendar,maximum_nights,int64,702,0,0.0



--- Profiling Snippet: REVIEWS (Top 5 Columns) ---


,Dataset,Column,DataType,Cardinality,NullCount,NullRate
0,reviews,listing_id,int64,11995,0,0.0
1,reviews,id,int64,992978,0,0.0
2,reviews,date,object,5102,0,0.0
3,reviews,reviewer_id,int64,940169,0,0.0
4,reviews,reviewer_name,object,121888,4,0.0



--- GLOBAL DATA QUALITY REPORT ---


,Dataset,Total Rows,Total Columns,Global Null Rate (%),Memory Footprint
0,listings,16107,90,20.39,75.90 MB
1,calendar,5879060,5,0.00,745.69 MB
2,reviews,992991,6,0.00,488.51 MB


### Detect and document duplicate records using deterministic and fuzzy matching techniques

In [56]:
# Deterministic Matching Technique
for name, df in raw_data.items():
    if df is not None:
        exact_dups = df.duplicated().sum()
        print(f" [{name.upper()}] found {exact_dups} exact row duplicates.")

 [LISTINGS] found 0 exact row duplicates.
 [CALENDAR] found 0 exact row duplicates.
 [REVIEWS] found 0 exact row duplicates.


In [57]:
# Fuzzy Matching Technique
if raw_data['listings'] is not None and 'name' in raw_data['listings']. columns:
    listings_df = raw_data['listings'].dropna(subset=['name']).copy()
    
    #sample batch of unique titles
    sample_titles = listings_df['name'].head(100).unique()
    fuzzy_matches = []
    
    
    for i, title_a in enumerate(sample_titles):
        for j, title_b in enumerate(sample_titles):
            if i < j:
                score = fuzz.ratio(title_a, title_b)
                # Flag variations that are nearly identical but contain typos/minor shifts
                if 85 <= score < 100:
                    fuzzy_matches.append((title_a, title_b, score))
                    
    fuzzy_df = pd.DataFrame(fuzzy_matches, columns=['Record Title A', 'Record Title B', 'Similarity Score'])
    print(f"   Potential Fuzzy/Near Duplicates Flagged: {len(fuzzy_df)}")
    if not fuzzy_df.empty:
        display(fuzzy_df.head(5))
print("\n" + "="*80 + "\n")


   Potential Fuzzy/Near Duplicates Flagged: 3


,Record Title A,Record Title B,Similarity Score
0,Sagrada Familia Clot Patio - Wifi/Private/Clean,Sagrada Familia Clot Atico - Wifi/Private/Clean,96
1,Sagrada Familia Clot Patio - Wifi/Private/Clean,Sagrada Familia Clot Vista - Wifi/Private/Clean,91
2,Sagrada Familia Clot Atico - Wifi/Private/Clean,Sagrada Familia Clot Vista - Wifi/Private/Clean,91


### Assess completeness

In [58]:
for name, prof_df in profiles.items():
    missing_sorted = prof_df.sort_values(by='NullRate', ascending=False).head(3)
    display(missing_sorted[['Column', 'NullCount', 'NullRate']])
    # print(prof_df.columns.tolist())
    

,Column,NullCount,NullRate
7,neighborhood_overview,16107,100.0
14,host_since,16107,100.0
21,host_response_time,16107,100.0


,Column,NullCount,NullRate
0,listing_id,0,0.0
1,date,0,0.0
2,available,0,0.0


,Column,NullCount,NullRate
5,comments,104,0.01
0,listing_id,0,0.00
1,id,0,0.00


### Identify outliers in key numerical fields

In [59]:
def detect_outliers(df,column):
    if df is None or column not in df.columns:
        return
    
    series = df[column].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    numeric_series = pd.to_numeric(series, errors='coerce').dropna()
    
    if len(numeric_series) == 0:
        return
    
    z_scores = np.abs(stats.zscore(numeric_series))
    outlies_mask = z_scores > 3
    outlier_count = np.sum(outlies_mask)
    
    print(f"Column '{column}': Flagged {outlier_count} outliers out of {len(numeric_series)} records (|Z| > 3).")
    if outlier_count > 0:
        top_extremes = numeric_series[outlies_mask].sort_values(ascending=False).head(3)
        print(f"Top Extreme Outlier Values: {list(top_extremes)}")
        
if raw_data['listings'] is not None:
    print("Outliers - listings")
    detect_outliers(raw_data['listings'], 'price')
    detect_outliers(raw_data['listings'], 'number_of_reviews')
    detect_outliers(raw_data['listings'], 'availability_365')
    
# if raw_data['calendar'] is not None:
#     print("Outliers - calendar")
#     detect_outliers(raw_data['calendar'], 'price')
#     detect_outliers(raw_data['calendar'], 'number_of_reviews')
#     detect_outliers(raw_data['calendar'], 'availability_365')
        

Outliers - listings
Column 'price': Flagged 196 outliers out of 12730 records (|Z| > 3).
Top Extreme Outlier Values: [10542.22, 7293.0, 7075.0]
Column 'number_of_reviews': Flagged 366 outliers out of 16107 records (|Z| > 3).
Top Extreme Outlier Values: [2405, 2091, 1922]
Column 'availability_365': Flagged 0 outliers out of 16107 records (|Z| > 3).


### Validate data against domain expecta<ons

In [60]:
def validate_domain_rules(datasets):
    listings = datasets.get('listings')
    
    if listings is not None:
        # Clean currency configurations for evaluation
        prices = listings['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
        prices = pd.to_numeric(prices, errors='coerce')
        
        # price validation
        negative_price_violations = (prices < 0).sum()
        
        # latitude violation
        if 'latitude' in listings.columns:
            invalid_lat_violations = (~listings['latitude'].between(-90, 90)).sum()
            
        # longtitude violation
        invalid_lon_violations = 0
        if 'longitude' in listings.columns:
            invalid_lon_violations = (~listings['longitude'].between(-180, 180)).sum()
            
        print(f"Negative Pricing Flagged: {negative_price_violations}")
        print(f"Out-of-Bounds Latitude (-90 to 90): {invalid_lat_violations}")
        print(f"Out-of-Bounds Longitude (-180 to 180): {invalid_lon_violations}")
        
        if negative_price_violations == 0 and invalid_lat_violations == 0 and invalid_lon_violations == 0:
            print("\n All ingested data validated")
        else:
            print("\n Invalidate parameters. Attention required")
            
validate_domain_rules(raw_data)
        
        
        
        

Negative Pricing Flagged: 0
Out-of-Bounds Latitude (-90 to 90): 0
Out-of-Bounds Longitude (-180 to 180): 0

 All ingested data validated


# 3.2 Data Cleaning & Standardization

In [61]:
# create a copy of raw data dictionary to stroe clean DataFrames
clean_data = {key: df.copy() if df is not None else None for key, df in raw_data.items()}



### Standardize price columns

In [62]:
def standardize_price(df, column_name):
    if df is not None and column_name in df.columns:
        # Force text format, strip whitespace, remove '$' and thousands separators ','
        cleaned_series = df[column_name].astype(str)\
                                        .str.replace('$', '', regex=False)\
                                        .str.replace(',', '', regex=False)\
                                        .str.strip()
        # Cast to float, coercing unparseable text anomalies to NaN
        df[column_name] = pd.to_numeric(cleaned_series, errors='coerce')
    return df
        
clean_data['listings'] = standardize_price(clean_data['listings'], 'price')
clean_data['calendar'] = standardize_price(clean_data['calendar'], 'price')
clean_data['calendar'] = standardize_price(clean_data['calendar'], 'adjusted_price')

# if clean_data['listings'] is not None and 'price' in clean_data['listings'].columns:
#     display(clean_data['listings'][['name', 'price']].head(5))
    



### Parse and standardize date fields

In [63]:
# listings date features
if clean_data['listings'] is not None:
    for date_col in ['last_scraped', 'calendar_last_scraped', 'host_since', 'first_review', 'last_review']:
        if date_col in clean_data['listings'].columns:
            clean_data['listings'][date_col] = pd.to_datetime(clean_data['listings'][date_col], errors='coerce', format='mixed')
            
# Calendar date features
if clean_data['calendar'] is not None and 'date' in clean_data['calendar'].columns:
    clean_data['calendar']['date'] = pd.to_datetime(clean_data['calendar']['date'], errors='coerce', format='mixed')

# Reviews date features
if clean_data['reviews'] is not None and 'date' in clean_data['reviews'].columns:
    clean_data['reviews']['date'] = pd.to_datetime(clean_data['reviews']['date'], errors='coerce', format='mixed')
    
 
# # Print results   
# if clean_data['listings'] is not None:
#     target_listing_dates = ['last_scraped', 'calendar_last_scraped', 'host_since', 'first_review', 'last_review']
#     available_listing_dates = [col for col in target_listing_dates if col in clean_data['listings'].columns]
    
#     for col in available_listing_dates:
#         print(f"Column '{col}': Data Type -> {clean_data['listings'][col].dtype}")
    
#     display(clean_data['listings'][available_listing_dates].dropna().head(3))
    

# if clean_data['calendar'] is not None and 'date' in clean_data['calendar'].columns:
#     display(clean_data['calendar'][['listing_id', 'date']].head(3))
    

# if clean_data['reviews'] is not None and 'date' in clean_data['reviews'].columns:
#     display(clean_data['reviews'][['listing_id', 'id', 'date']].head(3))
   



### Normalize free-text fields

In [64]:
if clean_data['listings'] is not None:
    categorical_targets = ['property_type', 'room_type']
    for categorical_col in categorical_targets:
        if categorical_col in clean_data['listings'].columns:
            clean_data['listings'][categorical_col] = clean_data['listings'][categorical_col].astype(str)\
                                                                             .str.lower()\
                                                                             .str.strip()\
                                                                             .replace('nan', np.nan) 
                                                                             
# # Print results                                                                            
# if clean_data['listings'] is not None:
#     categorical_targets = ['property_type', 'room_type']
    
#     for categorical_col in categorical_targets:
#         if categorical_col in clean_data['listings'].columns:
#             print(f"Unique Values Profile for '{categorical_col}'")
            
#             # Fetch a sample of unique values to prove they are lowercase and stripped
#             unique_samples = clean_data['listings'][categorical_col].dropna().unique()[:5]
#             print(f"Sample categories: {list(unique_samples)}")
            
#     available_cols = [c for c in categorical_targets if c in clean_data['listings'].columns]
#     display(clean_data['listings'][available_cols].head(10))
# else:
#     print("The listings dataset could not be found or loaded in clean_data.")                                                                         
            

### Handle missing values with appropriate strategies

In [65]:
if clean_data['listings'] is not None:
    # Sentinel values for critical missing categorical tracking
    if 'host_name' in clean_data['listings'].columns:
        clean_data['listings']['host_name'] = clean_data['listings']['host_name'].fillna('Unknown Host')
    # Numeric continuous variables backfilled via statistical medians
    review_score_columns = [col for col in clean_data['listings'].columns if 'review_scores' in col]
    for score_col in review_score_columns:
        median_val = clean_data['listings'][score_col].median()
        # Fallback to 0 if the entire column is empty
        if pd.isna(median_val): median_val = 0.0
        clean_data['listings'][score_col] = clean_data['listings'][score_col].fillna(median_val)
        
if clean_data['reviews'] is not None and 'comments' in clean_data['reviews'].columns:
    # Strategy C: Explicit empty strings for unstructured text fields (retains NLP integrity)
    clean_data['reviews']['comments'] = clean_data['reviews']['comments'].fillna('')   
    
    
    
# Print results

# Sentinel Value Replacement
if clean_data['listings'] is not None and 'host_name' in clean_data['listings'].columns:
    remaining_nans = clean_data['listings']['host_name'].isna().sum()
    sentinel_matches = (clean_data['listings']['host_name'] == 'Unknown Host').sum()
    
    print(f"Remaining NaNs in 'host_name': {remaining_nans}")
    print(f"Total rows safely recovered with 'Unknown Host': {sentinel_matches}")

# Numeric Matrix Imputation via Column Medians
if clean_data['listings'] is not None:
    
    review_score_columns = [col for col in clean_data['listings'].columns if 'review_scores' in col]
    
    if review_score_columns:
        imputation_audit = []
        for col in review_score_columns:
            # We calculate what the median value was for this run
            calculated_median = raw_data['listings'][col].median()
            if pd.isna(calculated_median): calculated_median = 0.0
            
            imputation_audit.append({
                'Review Metric Column': col,
                'Computed Median': round(calculated_median, 2),
                'Remaining NaNs': clean_data['listings'][col].isna().sum()
            })
        
        display(pd.DataFrame(imputation_audit))
    else:
        print(" -> No review score columns found in listings schema.")
    print("-" * 65 + "\n")


if clean_data['reviews'] is not None and 'comments' in clean_data['reviews'].columns:
    remaining_text_nans = clean_data['reviews']['comments'].isna().sum()
    empty_string_matches = (clean_data['reviews']['comments'] == '').sum()
    
    print(f"Remaining NaNs in 'comments': {remaining_text_nans}")
    print(f"Total empty rows safely protected as text blanks ('') for NLP: {empty_string_matches}")
    
    
        
    

Remaining NaNs in 'host_name': 0
Total rows safely recovered with 'Unknown Host': 125


,Review Metric Column,Computed Median,Remaining NaNs
0,review_scores_rating,4.72,0
1,review_scores_accuracy,4.78,0
2,review_scores_cleanliness,4.75,0
3,review_scores_checkin,4.85,0
4,review_scores_communication,4.86,0
5,review_scores_location,4.85,0
6,review_scores_value,4.58,0


-----------------------------------------------------------------

Remaining NaNs in 'comments': 0
Total empty rows safely protected as text blanks ('') for NLP: 104


### Remove or flag records that fail validation rule

In [66]:
if clean_data['listings'] is not None:
    initial_count = len(clean_data['listings'])
    
    # Isolate conditions for logical conformity
    valid_price = (clean_data['listings']['price'] >= 0) | (clean_data['listings']['price'].isna())
    valid_lat = clean_data['listings']['latitude'].between(-90, 90) if 'latitude' in clean_data['listings'].columns else True
    valid_lon = clean_data['listings']['longitude'].between(-180, 180) if 'longitude' in clean_data['listings'].columns else True
    
    # Keep only rows satisfying all criteria
    clean_data['listings'] = clean_data['listings'][valid_price & valid_lat & valid_lon]
    excised_count = initial_count - len(clean_data['listings'])
    print(f"Dropped {excised_count} corrupted rows failing structural domain constraints.")

Dropped 0 corrupted rows failing structural domain constraints.


### Standardize geographic fields

In [67]:
if clean_data['listings'] is not None:
    # Enforce coordinate rounding precision to match a ~11.1 meter grid index limit (4 decimals)
    if 'latitude' in clean_data['listings'].columns:
        clean_data['listings']['latitude'] = clean_data['listings']['latitude'].round(4)
    if 'longitude' in clean_data['listings'].columns:
        clean_data['listings']['longitude'] = clean_data['listings']['longitude'].round(4)
        
    # Standardize cross-record name formatting for spatial group operations
    if 'neighbourhood_cleansed' in clean_data['listings'].columns:
        # Title case makes it clean for production reporting views: e.g. "back bay" -> "Back Bay"
        clean_data['listings']['neighbourhood_cleansed'] = clean_data['listings']['neighbourhood_cleansed'].str.title()
        
    if clean_data['listings'] is not None:
        print("Print results for verification")
        columns_to_show = [col for col in ['neighbourhood_cleansed', 'price', 'latitude', 'longitude', 'host_since'] if col in clean_data['listings'].columns]
        display(clean_data['listings'][columns_to_show].head(5))

Print results for verification


,neighbourhood_cleansed,price,latitude,longitude,host_since
0,La Sagrada Família,334.00,41.4056,2.1726,NaT
1,El Besòs I El Maresme,452.67,41.4124,2.2198,NaT
2,El Barri Gòtic,344.64,41.3798,2.1762,NaT
3,La Barceloneta,42.75,41.3804,2.1909,NaT
4,La Dreta De L'Eixample,173.46,41.3963,2.1683,NaT


# 3.3 Data Enrichment & Joining

### Join listings with review summaries

In [68]:
if clean_data['reviews'] is not None and clean_data['listings'] is not None:

    review_summaries = clean_data['reviews'].groupby('listing_id').agg(
        pipeline_total_reviews=('id', 'count'),
        earliest_review_date=('date', 'min'),
        latest_review_date=('date', 'max')
    ).reset_index()
    
    # Merge findings into the master listings dataframe using an left relational join
    master_listings = pd.merge(
        clean_data['listings'], 
        review_summaries, 
        left_on='id', 
        right_on='listing_id', 
        how='left'
    )
    # Safely align missing join vectors to zero count indices
    master_listings['pipeline_total_reviews'] = master_listings['pipeline_total_reviews'].fillna(0).astype(int)
else:
    print("   Skipping review compilation: Datasets not uniformly instantiated.")
    master_listings = clean_data['listings'].copy() if clean_data['listings'] is not None else None
    
    
display(master_listings)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,listing_id,pipeline_total_reviews,earliest_review_date,latest_review_date
0,18674,https://www.airbnb.com/rooms/18674,20260321160757,2026-03-22,city scrape,Huge flat for 8 people close to Sagrada Familia,110m2 apartment to rent in Barcelona. Located ...,NaN,https://a0.muscache.com/pictures/13031453/413c...,71615,...,NaN,18,18,0,0,0.35,18674.0,55,2013-05-27,2026-03-02
1,23197,https://www.airbnb.com/rooms/23197,20260321160757,2026-03-21,city scrape,CCIB Forum· Large Balcony · 5 min walk to CCIB,"Beautiful, spacious apartment with large balco...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,90417,...,NaN,1,1,0,0,0.51,23197.0,93,2011-03-15,2026-03-06
2,34981,https://www.airbnb.com/rooms/34981,20260321160757,2026-03-21,city scrape,VIDRE HOME PLAZA REAL on LAS RAMBLAS,Spacious apartment for large families or group...,NaN,https://a0.muscache.com/pictures/c4d1723c-e479...,73163,...,NaN,2,2,0,0,1.56,34981.0,294,2010-10-03,2026-03-02
3,36763,https://www.airbnb.com/rooms/36763,20260321160757,2026-03-22,previous scrape,In front of the beach,NaN,NaN,https://a0.muscache.com/pictures/airflow/Hosti...,158596,...,NaN,1,0,1,0,0.61,36763.0,108,2011-09-28,2023-12-13
4,40983,https://www.airbnb.com/rooms/40983,20260321160757,2026-03-22,city scrape,Soho Colonial Eclectic Apartment,Our cool and stylish Loft Classical Apartment ...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,177617,...,NaN,5,2,3,0,2.41,40983.0,434,2011-06-16,2026-03-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16102,1645109330476648794,https://www.airbnb.com/rooms/1645109330476648794,20260321160757,2026-03-21,city scrape,3 bedroom apartment next to Plaza,The 3-bedroom holiday apartment next to Plaza ...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,686781509,...,NaN,1,1,0,0,NaN,NaN,0,NaT,NaT
16103,1645560017838329398,https://www.airbnb.com/rooms/1645560017838329398,20260321160757,2026-03-21,city scrape,Joan Blanques 3 1,This cozy 3-bedroom apartment is located on th...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,4396136,...,NaN,13,13,0,0,NaN,NaN,0,NaT,NaT
16104,1645573548092816588,https://www.airbnb.com/rooms/1645573548092816588,20260321160757,2026-03-21,city scrape,Joan Blanques 3 1,This cozy 3-bedroom apartment is located on th...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,86551441,...,NaN,15,15,0,0,NaN,NaN,0,NaT,NaT
16105,1645594736674702639,https://www.airbnb.com/rooms/1645594736674702639,20260321160757,2026-03-21,city scrape,Habitación privada en Barcelona,"Ideal room for two people 🛏️✨, to disconnect a...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,689765133,...,NaN,1,0,1,0,NaN,NaN,0,NaT,NaT


### Integrate calendar data to compute per-listing occupancy rates and revenue estimates

In [69]:
if clean_data.get('calendar') is not None and master_listings is not None:
    
    # Create a clean working copy of calendar data
    calendar_df = clean_data['calendar'].copy()
    
    # Map availability text flags to binary numbers ('f' means blocked/occupied = 1)
    calendar_df['is_occupied'] = calendar_df['available'].map({'t': 0, 'f': 1}).fillna(0)
    
    # ompress the 365 daily rows down into a single row per unique property listing
    calendar_metrics = calendar_df.groupby('listing_id').agg(
        tracked_days=('date', 'count'),
        booked_days=('is_occupied', 'sum')
    ).reset_index()
    
    # Calculate the Occupancy Rate percentage safely (preventing division-by-zero)
    calendar_metrics['computed_occupancy_rate (%)'] = np.where(
        calendar_metrics['tracked_days'] > 0,
        (calendar_metrics['booked_days'] / calendar_metrics['tracked_days']) * 100,
        0.0
    ).round(2)
    
    # Merge these occupancy metrics back into your master listings layer
    master_listings = pd.merge(
        master_listings, 
        calendar_metrics[['listing_id', 'tracked_days', 'booked_days', 'computed_occupancy_rate (%)']], 
        left_on='id', 
        right_on='listing_id', 
        how='left'
    ).drop(columns=['listing_id'], errors='ignore')
    
    # REVENUE ESTIMATE CALCULATOR (Cross-Table Multiplication Strategy)
    # Since the calendar has no daily price, use: (Booked Days * Listings Base Price)

    if 'price' in master_listings.columns:
        master_listings['estimated_calendar_revenue'] = (
            master_listings['booked_days'].fillna(0) * master_listings['price'].fillna(0)
        ).round(2)
    else:
        master_listings['estimated_calendar_revenue'] = 0.0
        print("WARNING: 'price' column was missing in master_listings. revenue set to 0.0.")
        
    print("\nSUCCESS:")
    
    # Display verification preview
    preview_cols = ['id', 'name', 'price', 'tracked_days', 'booked_days', 'computed_occupancy_rate (%)', 'estimated_calendar_revenue']
    safe_preview = [col for col in preview_cols if col in master_listings.columns]
    display(master_listings[safe_preview].head(5))

else:
    print("ERROR")


SUCCESS:


,id,name,price,tracked_days,booked_days,computed_occupancy_rate (%),estimated_calendar_revenue
0,18674,Huge flat for 8 people close to Sagrada Familia,334.00,365,254,69.59,84836.00
1,23197,CCIB Forum· Large Balcony · 5 min walk to CCIB,452.67,365,135,36.99,61110.45
2,34981,VIDRE HOME PLAZA REAL on LAS RAMBLAS,344.64,365,83,22.74,28605.12
3,36763,In front of the beach,42.75,365,276,75.62,11799.00
4,40983,Soho Colonial Eclectic Apartment,173.46,365,92,25.21,15958.32


### Enrich listings with neighbourhood-level aggregates

In [73]:
if master_listings is not None and 'neighbourhood_cleansed' in master_listings.columns:
    
    # 1. Determine the best available review column name in your schema
    # Common variations include 'review_scores_rating' or 'review_scores_value'
    possible_rating_cols = ['review_scores_rating', 'review_scores_value']
    rating_col = next((col for col in possible_rating_cols if col in master_listings.columns), None)
    
    # 2. Define the mathematical aggregation rules for our spatial neighborhoods
    # Count listing IDs to get density, find the middle point (median) for pricing
    agg_rules = {
        'id': 'count',
        'price': 'median'
    }
    
    # If a valid review score column exists, calculate its arithmetic mean
    if rating_col:
        agg_rules[rating_col] = 'mean'
        
    # 3. Compute the macro statistical metrics table
    nbr_stats = master_listings.groupby('neighbourhood_cleansed').agg(agg_rules).reset_index()
    
    # 4. Standardize column names to prevent confusion with individual listing columns
    rename_map = {
        'id': 'nbr_listing_density',
        'price': 'nbr_median_price'
    }
    if rating_col:
        rename_map[rating_col] = 'nbr_avg_review_score'
        
    nbr_stats = nbr_stats.rename(columns=rename_map)
    
    # 5. Map the neighborhood aggregate metrics back into every single listing record
    master_listings = pd.merge(
        master_listings, 
        nbr_stats, 
        on='neighbourhood_cleansed', 
        how='left'
    )
    

    
    # 6. Display confirmation preview showing individual vs neighborhood metrics
    preview_cols = ['id', 'neighbourhood_cleansed', 'price', 'nbr_median_price', 'nbr_listing_density']
    if rating_col:
        preview_cols.extend([rating_col, 'nbr_avg_review_score'])
        
    display(master_listings[preview_cols].head(5))

else:
    print("ERROR")

,id,neighbourhood_cleansed,price,nbr_median_price,nbr_listing_density,review_scores_rating,nbr_avg_review_score
0,18674,La Sagrada Família,334.00,238.565,915,4.37,4.611104
1,23197,El Besòs I El Maresme,452.67,134.370,106,4.83,4.649245
2,34981,El Barri Gòtic,344.64,124.605,961,4.58,4.601082
3,36763,La Barceloneta,42.75,83.135,426,4.79,4.690915
4,40983,La Dreta De L'Eixample,173.46,230.000,2198,4.54,4.659641


### Derive calculated fields: host tenure (years on plakorm), review frequency, price-per-bedroom.

In [71]:
if master_listings is not None:
    if 'host_since' in master_listings.columns:
        # Establish accurate current milestone anchor (June 2026)
        pipeline_reference_date = pd.Timestamp('2026-06-23') 
        
        # Ensure host_since is parsed as a true timestamp sequence
        host_datetime = pd.to_datetime(master_listings['host_since'], errors='coerce')
        host_delta = pipeline_reference_date - host_datetime
        
        # Convert daily differences to decimal years
        master_listings['derived_host_tenure_years'] = (host_delta.dt.days / 365.25).round(2)
        master_listings['derived_host_tenure_years'] = master_listings['derived_host_tenure_years'].fillna(0.0)
    else:
        master_listings['derived_host_tenure_years'] = 0.0
        print(" 'host_since' column missing. tenure initialized to 0.0.")

    if 'number_of_reviews' in master_listings.columns:
        # Calculate reviews per year; np.where handles division-by-zero for brand new accounts
        master_listings['derived_reviews_per_year'] = np.where(
            master_listings['derived_host_tenure_years'] > 0,
            (master_listings['number_of_reviews'] / master_listings['derived_host_tenure_years']).round(2),
            0.0
        )
    else:
        master_listings['derived_reviews_per_year'] = 0.0

    if 'price' in master_listings.columns:
        # Fallback to 1 for studios or missing bedroom counts to prevent division anomalies
        if 'bedrooms' in master_listings.columns:
            effective_bedrooms = master_listings['bedrooms'].fillna(1).replace(0, 1)
        else:
            effective_bedrooms = 1
            
        master_listings['derived_price_per_bedroom'] = (master_listings['price'] / effective_bedrooms).round(2)
    else:
        master_listings['derived_price_per_bedroom'] = 0.0

    
    # Display preview verification matrix
    preview_cols = ['id', 'host_since', 'derived_host_tenure_years', 'number_of_reviews', 'derived_reviews_per_year', 'price', 'derived_price_per_bedroom']
    safe_preview = [col for col in preview_cols if col in master_listings.columns]
    display(master_listings[safe_preview].head(5))

else:
    print("ERROR")

,id,host_since,derived_host_tenure_years,number_of_reviews,derived_reviews_per_year,price,derived_price_per_bedroom
0,18674,NaT,0.0,55,0.0,334.00,111.33
1,23197,NaT,0.0,93,0.0,452.67,150.89
2,34981,NaT,0.0,294,0.0,344.64,86.16
3,36763,NaT,0.0,108,0.0,42.75,42.75
4,40983,NaT,0.0,434,0.0,173.46,173.46
